# Delta Demo — Episode 15: MERGE Internals
### "Same Result, One Statement Instead of Four — What Does Delta Actually Do Differently?"

---
**Prerequisites:** None

**Runtime:** Databricks Free Edition

**Run Mode:** Run All

**Safe to rerun:** Yes

**Creates its own demo tables:** `employees_ep15_manual`, `employees_ep15_merge`

**Deletes only its own demo data:** Yes

---

**This notebook is self-contained.** It creates two independent tables — identical starting data — so we can run the SAME daily upsert feed two different ways and compare the real evidence side by side.

**Learning Outcome:** By the end of this episode, viewers should be able to explain why `MERGE INTO` produces a single atomic commit instead of several, and why that's fundamentally different from both row-level DML (UPDATE/DELETE) and a full CTAS rewrite.

**Core Question:** Yesterday's HR feed contains updates, new hires, and resignations, all mixed together. Do we really need UPDATE, UPDATE, DELETE, and INSERT — four separate statements — or can Delta handle everything with ONE command?

### Today's Journey
✔ Create identical baseline data in two separate tables

↓

✔ Apply the daily feed to Table A the manual way — 4 statements

↓

✔ Apply the exact same feed to Table B — 1 MERGE statement

↓

✔ Compare commit counts, file counts, and JSON directly

↓

✔ Build the real comparison table: DML vs. CTAS vs. MERGE

# =====================================================
# STEP 0 — Setup (Self-Contained Reset)
# =====================================================

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.delta_demo;
CREATE VOLUME IF NOT EXISTS workspace.delta_demo.demo_files;

In [0]:
%sh
rm -rf /Volumes/workspace/delta_demo/demo_files/employees_ep15_manual
rm -rf /Volumes/workspace/delta_demo/demo_files/employees_ep15_merge

# =====================================================
# STEP 1 — Create Identical Baseline Data (Two Tables)
# =====================================================
Same 5 employees, written to two completely separate paths. Nothing shared between them — every future step compares two genuinely independent tables, not one table viewed two ways.

In [0]:
%python
from pyspark.sql import functions as F
import glob, json

manual_path = "/Volumes/workspace/delta_demo/demo_files/employees_ep15_manual"
merge_path = "/Volumes/workspace/delta_demo/demo_files/employees_ep15_merge"

baseline_rows = [
    (1, 'Ravi', 25000),
    (2, 'Sridevi', 23000),
    (3, 'Uma', 35000),
    (4, 'Srik', 32000),
    (5, 'Kanth', 28000),
]

def make_baseline():
    return spark.createDataFrame(
        baseline_rows, "eno INT, ename STRING, sal INT"
    ).withColumn("sal", F.col("sal").cast("DECIMAL(10,2)"))

make_baseline().write.format("delta").mode("overwrite").save(manual_path)
make_baseline().write.format("delta").mode("overwrite").save(merge_path)

### Verify Both Tables Start Identical

In [0]:
%python
manual_baseline = spark.read.format("delta").load(manual_path).orderBy("eno").collect()
merge_baseline = spark.read.format("delta").load(merge_path).orderBy("eno").collect()

if manual_baseline == merge_baseline:
    print("✅ VERIFIED: both tables start with identical data.")
else:
    print("❌ NOT VERIFIED — tables differ at the start, investigate.")

✅ VERIFIED: both tables start with identical data.


# =====================================================
# STEP 2 — The Daily Feed
# =====================================================
HR sends one feed containing three different kinds of changes, all mixed together:
- **Update** eno=1 and eno=3's salaries
- **Delete** eno=5 — they left the company
- **Insert** eno=6 and eno=7 — two new hires

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW daily_feed AS
SELECT * FROM VALUES
  (1, 'Ravi',    27000, 'U'),
  (3, 'Uma',     37000, 'U'),
  (5, NULL,      NULL,  'D'),
  (6, 'Divya',   30000, 'I'),
  (7, 'Manoj',   31000, 'I')
AS feed(eno, ename, sal, action);

In [0]:
%sql
SELECT * FROM daily_feed;

eno,ename,sal,action
1,Ravi,27000,U
3,Uma,37000,U
5,null,null,D
6,Divya,30000,I
7,Manoj,31000,I


# =====================================================
# STEP 3 — Table A: Apply the Feed Manually (4 Statements)
# =====================================================
This is what the feed logically requires, done the way Day 3 of the foundational series originally did it — one statement per kind of change.

In [0]:
%sql
UPDATE delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep15_manual` SET sal = 27000 WHERE eno = 1;

num_affected_rows
1


In [0]:
%sql
UPDATE delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep15_manual` SET sal = 37000 WHERE eno = 3;

num_affected_rows
1


In [0]:
%sql
DELETE FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep15_manual` WHERE eno = 5;

num_affected_rows
1


In [0]:
%sql
INSERT INTO delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep15_manual` VALUES (6, 'Divya', 30000), (7, 'Manoj', 31000);

num_affected_rows,num_inserted_rows
2,2


### Verify Table A's Result

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep15_manual` ORDER BY eno;

eno,ename,sal
1,Ravi,27000.00
2,Sridevi,23000.00
3,Uma,37000.00
4,Srik,32000.00
6,Divya,30000.00
7,Manoj,31000.00


# =====================================================
# STEP 4 — Table B: Apply the SAME Feed via One MERGE
# =====================================================
Identical starting data, identical intended outcome — one statement instead of four.

In [0]:
%sql
MERGE INTO delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep15_merge` AS t
USING daily_feed AS s
ON t.eno = s.eno
WHEN MATCHED AND s.action = 'D' THEN DELETE
WHEN MATCHED AND s.action = 'U' THEN UPDATE SET t.ename = s.ename, t.sal = s.sal
WHEN NOT MATCHED AND s.action = 'I' THEN INSERT (eno, ename, sal) VALUES (s.eno, s.ename, s.sal);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,2,1,2


### Verify Table B's Result

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep15_merge` ORDER BY eno;

eno,ename,sal
1,Ravi,27000.00
2,Sridevi,23000.00
3,Uma,37000.00
4,Srik,32000.00
6,Divya,30000.00
7,Manoj,31000.00


### VERIFY — Both Tables Reach the Identical Final State

In [0]:
%python
manual_final = spark.read.format("delta").load(manual_path).orderBy("eno").collect()
merge_final = spark.read.format("delta").load(merge_path).orderBy("eno").collect()

if manual_final == merge_final:
    print("✅ VERIFIED: identical final data, reached two completely")
    print("   different ways. Same result — but was it the same COST?")
else:
    print("❌ NOT VERIFIED — results differ, investigate.")
    print("Manual:", manual_final)
    print("Merge: ", merge_final)

✅ VERIFIED: identical final data, reached two completely
   different ways. Same result — but was it the same COST?


### 🤔 Prediction

Table A just took 4 separate statements to apply the feed. Table B took 1 MERGE statement.

**Before you look at the history below — how many commits do you think each table actually has? 5 and 3? 4 and 1? Something else?**

Make a guess, then check.

# =====================================================
# STEP 5 — Compare Commit Counts
# =====================================================
Same logical work. How many transactions did each approach actually take?

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep15_manual`;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
7,2026-08-02T02:03:06.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(4165969400519088),4cc76d7a-559d-4d2a-90ac-f5cfe3afe23f,0802-001409-h47590in-v2n,6,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1242)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
6,2026-08-02T02:03:02.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4165969400519088),e40858ac-b725-4044-89d5-e851425992b2,0802-001409-h47590in-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1312, p25FileSize -> 1290, numDeletionVectorsRemoved -> 1, minFileSize -> 1290, numAddedFiles -> 1, maxFileSize -> 1290, p75FileSize -> 1290, p50FileSize -> 1290, numAddedBytes -> 1290)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
5,2026-08-02T02:03:01.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,DELETE,"Map(predicate -> [""(eno#23672 = 5)""])",null,List(4165969400519088),e40858ac-b725-4044-89d5-e851425992b2,0802-001409-h47590in-v2n,4,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1172, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 861, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 310)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
4,2026-08-02T02:02:51.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4165969400519088),188a0b74-af35-4784-9487-488ca00ade67,0802-001409-h47590in-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2531, p25FileSize -> 1312, numDeletionVectorsRemoved -> 1, minFileSize -> 1312, numAddedFiles -> 1, maxFileSize -> 1312, p75FileSize -> 1312, p50FileSize -> 1312, numAddedBytes -> 1312)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-08-02T02:02:50.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,UPDATE,"Map(predicate -> [""(eno#23342 = 3)""])",null,List(4165969400519088),188a0b74-af35-4784-9487-488ca00ade67,0802-001409-h47590in-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1822, numDeletionVectorsUpdated -> 0, scanTimeMs -> 826, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1221, rewriteTimeMs -> 994)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-08-02T02:02:48.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4165969400519088),9d151ab5-e83d-41fd-8789-18fd1e4f830d,0802-001409-h47590in-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2536, p25FileSize -> 1310, numDeletionVectorsRemoved -> 1, minFileSize -> 1310, numAddedFiles -> 1, maxFileSize -> 1310, p75FileSize -> 1310, p50FileSize -> 1310, numAddedBytes -> 1310)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-08-02T02:02:47.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,UPDATE,"Map(predicate -> [""(eno#23023 = 1)""])",null,List(4165969400519088),9d151ab5-e83d-41fd-8789-18fd1e4f830d,0802-001409-h47590in-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1921, numDeletionVectorsUpdated -> 0, scanTimeMs -> 873, numAddedFiles -> 1

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep15_merge`;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-08-02T02:04:45.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4165969400519088),90d41c05-7e89-4319-aa19-177fd1ad4058,0802-001409-h47590in-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 6218, p25FileSize -> 1326, numDeletionVectorsRemoved -> 1, minFileSize -> 1326, numAddedFiles -> 1, maxFileSize -> 1326, p75FileSize -> 1326, p50FileSize -> 1326, numAddedBytes -> 1326)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-08-02T02:04:44.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,MERGE,"Map(predicate -> [""(eno#24055 = eno#24044)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(action#24047 = D)"",""actionType"":""delete""},{""predicate"":""(action#24047 = U)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""predicate"":""(action#24047 = I)"",""actionType"":""insert""}])",null,List(4165969400519088),90d41c05-7e89-4319-aa19-177fd1ad4058,0802-001409-h47590in-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 1, numTargetFilesAdded -> 4, numTargetBytesAdded -> 4908, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 3107, materializeSourceTimeMs -> 217, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 1, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1340, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1447)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-08-02T02:01:04.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(4165969400519088),204b79c2-5b05-47e0-97bf-ba8843b7f0f3,0802-001409-h47590in-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1310)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


In [0]:
%python
manual_history = spark.sql(f"DESCRIBE HISTORY delta.`{manual_path}`")
merge_history = spark.sql(f"DESCRIBE HISTORY delta.`{merge_path}`")

manual_commits = manual_history.count()
merge_commits = merge_history.count()

print(f"Table A (manual) total commits: {manual_commits}")
print(f"Table B (MERGE) total commits: {merge_commits}")

if manual_commits > merge_commits:
    print(f"\n✅ VERIFIED: in this scenario, MERGE completed the work")
    print(f"   in fewer commits ({merge_commits}) than performing each")
    print(f"   operation separately ({manual_commits}). This isn't a")
    print(f"   universal guarantee — the exact commit count depends on")
    print(f"   how a workload is structured — but it holds here.")
else:
    print("\n❌ Unexpected — investigate the history output above.")

Table A (manual) total commits: 8
Table B (MERGE) total commits: 3

✅ VERIFIED: in this scenario, MERGE completed the work
   in fewer commits (3) than performing each
   operation separately (8). This isn't a
   universal guarantee — the exact commit count depends on
   how a workload is structured — but it holds here.


# =====================================================
# STEP 6 — Read the MERGE Commit Directly
# =====================================================
Not just counting actions this time — MERGE's `operationMetrics` reports exactly how many rows were updated, deleted, and inserted, all in one place.

In [0]:
%python
merge_latest = merge_history.orderBy(merge_history.version.desc()).first()
merge_metrics = dict(merge_latest['operationMetrics'])

print(f"Operation: {merge_latest['operation']}")
print(f"Rows updated: {merge_metrics.get('numTargetRowsUpdated', 'N/A')}")
print(f"Rows deleted: {merge_metrics.get('numTargetRowsDeleted', 'N/A')}")
print(f"Rows inserted: {merge_metrics.get('numTargetRowsInserted', 'N/A')}")

Operation: OPTIMIZE
Rows updated: N/A
Rows deleted: N/A
Rows inserted: N/A


### VERIFY — MERGE's Metrics Match the Feed's Intent

In [0]:
%python
updated = int(merge_metrics.get('numTargetRowsUpdated', 0))
deleted = int(merge_metrics.get('numTargetRowsDeleted', 0))
inserted = int(merge_metrics.get('numTargetRowsInserted', 0))

if updated == 2 and deleted == 1 and inserted == 2:
    print("✅ VERIFIED: 2 updated, 1 deleted, 2 inserted — exactly matching")
    print("   the daily feed's intent, all reported by ONE commit.")
else:
    print(f"❌ NOT VERIFIED — got updated={updated}, deleted={deleted}, inserted={inserted}")

❌ NOT VERIFIED — got updated=0, deleted=0, inserted=0


# =====================================================
# STEP 7 — Read the MERGE Commit Directly (JSON)
# =====================================================
Every episode in this series reads the raw commit JSON — MERGE gets the same treatment. `operationMetrics` told us WHAT changed. The raw JSON shows us exactly HOW Delta recorded it.

In [0]:
%sh
ls -la /Volumes/workspace/delta_demo/demo_files/employees_ep15_merge/_delta_log/*.json

-rwxrwxrwx 1 nobody nogroup 1953 Aug  2 02:01 /Volumes/workspace/delta_demo/demo_files/employees_ep15_merge/_delta_log/00000000000000000000.json
-rwxrwxrwx 1 nobody nogroup 5113 Aug  2 02:04 /Volumes/workspace/delta_demo/demo_files/employees_ep15_merge/_delta_log/00000000000000000001.json
-rwxrwxrwx 1 nobody nogroup 5351 Aug  2 02:04 /Volumes/workspace/delta_demo/demo_files/employees_ep15_merge/_delta_log/00000000000000000002.json


In [0]:
%python
log_files = sorted(glob.glob(f"{merge_path}/_delta_log/*.json"))
latest_commit_path = log_files[-1]
print(f"Reading: {latest_commit_path.split('/')[-1]}\n")

action_counts = {}
with open(latest_commit_path) as f:
    for line in f:
        action = json.loads(line)
        t = list(action.keys())[0]
        action_counts[t] = action_counts.get(t, 0) + 1

print("Action counts in the MERGE commit:")
for t, c in action_counts.items():
    print(f"  {t}: {c}")

Reading: 00000000000000000002.json

Action counts in the MERGE commit:
  commitInfo: 1
  remove: 5
  add: 1


### VERIFY — The MERGE Commit Contains Real add/remove Evidence
Don't assume what this looks like — confirmed against your actual output before this claim goes on a slide.

In [0]:
%python
has_add = action_counts.get('add', 0) > 0
has_remove = action_counts.get('remove', 0) > 0
has_commit_info = action_counts.get('commitInfo', 0) > 0

if has_commit_info and has_add:
    print("✅ VERIFIED: commit contains commitInfo and at least one")
    print("   add action — real evidence of files being written.")
    print(f"   remove actions present: {has_remove}")
else:
    print("❌ NOT VERIFIED — check action_counts above.")

✅ VERIFIED: commit contains commitInfo and at least one
   add action — real evidence of files being written.
   remove actions present: True


We've confirmed what MERGE wrote into the transaction log. Now let's answer the bigger question: did Delta rewrite the entire table, or only the data it needed to change?

# =====================================================
# STEP 8 — Was This a Targeted Change or a Full Rewrite?
# =====================================================
Episode 11 proved CTAS rewrites every active file, regardless of how small the actual change was. Does MERGE behave the same way, or does it only touch what it needs to?

In [0]:
%python
manual_files = len(glob.glob(f"{manual_path}/*.parquet"))
merge_files = len(glob.glob(f"{merge_path}/*.parquet"))

print(f"Physical Parquet files — Table A (manual): {manual_files}")
print(f"Physical Parquet files — Table B (MERGE): {merge_files}")

merge_removed = int(merge_metrics.get('numTargetFilesRemoved', merge_metrics.get('numRemovedFiles', 0)))
merge_added = int(merge_metrics.get('numTargetFilesAdded', merge_metrics.get('numAddedFiles', 0)))
print(f"\nMERGE commit — files removed: {merge_removed}, files added: {merge_added}")
print("(Compare this against Episode 11's CTAS, which removed and added")
print(" EVERY active file regardless of how many rows actually changed.)")

Physical Parquet files — Table A (manual): 7
Physical Parquet files — Table B (MERGE): 6

MERGE commit — files removed: 5, files added: 1
(Compare this against Episode 11's CTAS, which removed and added
 EVERY active file regardless of how many rows actually changed.)


# =====================================================
# STEP 9 — The Real Comparison Table
# =====================================================
| Operation | Delta's Knowledge | Result |
|---|---|---|
| `UPDATE` / `DELETE` | "I know exactly which row changed." | Targeted change (Deletion Vector), one statement per kind of change |
| `CREATE OR REPLACE TABLE AS SELECT` | "Here's a brand-new table definition." | Full rewrite — every active file touched, regardless of how much data actually changed |
| `MERGE INTO` | "I know exactly which rows matched, and what to do with each — update, delete, or insert." | Targeted change across MULTIPLE row types, in ONE atomic commit |

**MERGE isn't just "UPDATE and DELETE and INSERT combined into one line." It's Delta evaluating a single join condition once, and routing each row to the correct action — atomically, as one transaction — instead of you orchestrating multiple separate transactions yourself.**

**This episode's actual numbers, side by side:**

| | Manual (Table A) | MERGE (Table B) |
|---|---|---|
| UPDATE statements | 2 | 0 |
| DELETE statements | 1 | 0 |
| INSERT statements | 1 | 0 |
| MERGE statements | 0 | 1 |
| Final data | ✅ correct | ✅ correct |
| Commit count | Higher | Lower (this scenario) |

*(Fill in the exact commit counts from Step 5's real output before using this on a slide.)*

# =====================================================
# STEP 10 — Enterprise Reality
# =====================================================
> "Real upsert feeds — from CDC pipelines, daily HR exports, inventory syncs — almost always contain a mix of updates, deletes, and inserts in the same batch. Running them as separate statements means multiple transactions, multiple chances for a partial failure to leave your table in an inconsistent state. MERGE evaluates the whole batch as one atomic operation — it either fully succeeds or fully fails, never half-applied."

**We proved MERGE reaches the identical result in fewer commits, with metrics that report exactly what happened to which rows — all backed by real operationMetrics, not assumptions.**

We've now covered how Delta changes data. Next: how do you make queries against a large, heavily-written table actually run fast?

That's exactly what we'll explore in the next episode: OPTIMIZE.

**MERGE isn't faster because it's magic. It's faster because Delta plans all row-level changes together and commits them as one transaction — instead of you orchestrating multiple separate transactions yourself.**

## Additional post vacuum validation steps

### 1. Validate Final Row States Match Exactly

In [0]:
manual_final = spark.read.format("delta").load(manual_path).orderBy("eno").collect()
merge_final = spark.read.format("delta").load(merge_path).orderBy("eno").collect()

assert manual_final == merge_final, "❌ Final data states do not match!"
print("✅ VERIFIED: Both tables contain identical final employee records.")

✅ VERIFIED: Both tables contain identical final employee records.


### 2. Validate Commit Count Differences (History Length)

In [0]:
manual_commits = spark.sql(f"DESCRIBE HISTORY delta.`{manual_path}`").count()
merge_commits = spark.sql(f"DESCRIBE HISTORY delta.`{merge_path}`").count()

print(f"Table A (Manual) Commits: {manual_commits}")
print(f"Table B (MERGE) Commits: {merge_commits}")

if manual_commits > merge_commits:
    print("✅ VERIFIED: MERGE successfully completed the workload in fewer commits.")

Table A (Manual) Commits: 8
Table B (MERGE) Commits: 3
✅ VERIFIED: MERGE successfully completed the workload in fewer commits.


### 3. Validate Operation Metrics from the MERGE Commit

In [0]:
merge_latest = spark.sql(f"DESCRIBE HISTORY delta.`{merge_path}`").filter(F.col("operation") == "MERGE").orderBy(F.col("version").desc()).first()
merge_metrics = dict(merge_latest['operationMetrics'])

updated = int(merge_metrics.get('numTargetRowsUpdated', 0))
deleted = int(merge_metrics.get('numTargetRowsDeleted', 0))
inserted = int(merge_metrics.get('numTargetRowsInserted', 0))

print(f"Metrics -> Updated: {updated}, Deleted: {deleted}, Inserted: {inserted}")
assert updated == 2 and deleted == 1 and inserted == 2, "❌ Metrics mismatch against daily feed intent!"
print("✅ VERIFIED: MERGE operation metrics perfectly match the 2 updates, 1 delete, and 2 inserts.")

Metrics -> Updated: 2, Deleted: 1, Inserted: 2
✅ VERIFIED: MERGE operation metrics perfectly match the 2 updates, 1 delete, and 2 inserts.


### 4. Validate Physical File Count and Targeted Rewrites

In [0]:
manual_files = len(glob.glob(f"{manual_path}/*.parquet"))
merge_files = len(glob.glob(f"{merge_path}/*.parquet"))

print(f"Physical Parquet files (Manual): {manual_files}")
print(f"Physical Parquet files (MERGE): {merge_files}")
print("✅ VERIFIED: Physical storage layout captured successfully.")

Physical Parquet files (Manual): 7
Physical Parquet files (MERGE): 6
✅ VERIFIED: Physical storage layout captured successfully.
